# Lección 1.6 — "No hay almuerzo gratis": por qué no existe un modelo perfecto

> **Módulo 1 · ¿Qué es aprender para una máquina?** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 60–75 min (lección con ejercicios intercalados ≈ 40 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 1.2 (vocabulario) y 1.5 (ciclo de vida, línea base).

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con tus palabras el teorema de **"no hay almuerzo gratis"** (*No Free Lunch*).
- Entender que cada modelo trae **supuestos** (su *sesgo inductivo*) y que gana cuando esos supuestos encajan con los datos.
- Comprobar con código que el modelo ganador **cambia** según los datos.
- Adoptar el hábito profesional: **probar varios modelos, comparar con una línea base y validar**.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números o palabras.

## ▶️ Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles, load_breast_cancer, load_wine, load_digits
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print("✅ Todo listo")

---
## 1. 📖 La intuición: no existe la herramienta para todo

En una caja de herramientas no hay "la mejor herramienta". El martillo es buenísimo con clavos y malísimo con tornillos. El destornillador, al revés. Preguntar "¿cuál es mejor?" no tiene sentido sin decir **para qué**.

Con los modelos de ML pasa lo mismo. En 1996–1997, David Wolpert y William Macready demostraron un resultado con un nombre muy gráfico: **"no hay almuerzo gratis"** (*No Free Lunch*). Dicho en simple:

> **Promediando sobre todos los problemas posibles, ningún modelo es mejor que otro.** Si un modelo gana en un tipo de problema, lo paga perdiendo en otro.

¿Por qué ganan algunos modelos en la práctica? Porque los problemas **reales** no son "todos los problemas posibles": tienen estructura, y un modelo gana cuando sus **supuestos** encajan con esa estructura.

### ✋ Ejercicio 1
Un blog titula: *"El algoritmo X es el mejor modelo de Machine Learning"*. Según el teorema, ¿qué le falta a esa frase para tener sentido?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Le falta **"…para este tipo de datos / este problema"**. Sin decir **en qué problema** y **con qué métrica**, la frase no significa nada. Un modelo puede ser el mejor para datos tabulares grandes y uno mediocre para pocas filas, imágenes o datos con otra estructura.
</details>

---
## 2. 📖 El concepto: cada modelo es una "apuesta" sobre el mundo

Con un número finito de ejemplos, hay infinitas reglas que encajan con ellos. Para elegir una, cada modelo **apuesta** por cierto tipo de regla. A esa apuesta se le llama **sesgo inductivo** (*inductive bias*): los supuestos que el modelo hace para poder generalizar.

| Modelo | Su apuesta (supuesto) | Gana cuando… | Sufre cuando… |
|---|---|---|---|
| **Regresión logística / lineal** (Lecciones 5.1–5.2) | La frontera (o la relación) es una **línea recta** | Muchas señales pequeñas que se **suman** | La frontera es curva o en forma de "islas" |
| **Árbol de decisión** (Lección 5.4) | La respuesta cambia en **escalones** según preguntas de una variable a la vez | Hay **reglas** con umbrales ("si ingreso > X y deuda < Y…") | La frontera es diagonal o suave |
| **KNN** (Lección 5.3) | Los ejemplos **cercanos** tienen la misma respuesta | Fronteras de forma rara, pocas variables, suficientes datos | Muchas variables irrelevantes (las distancias pierden sentido, Lección 2.8) |

### ✋ Ejercicio 2 — Empareja
¿Qué modelo apostarías que funciona mejor en cada caso?
1. Un banco aprueba préstamos con reglas internas del tipo "si ingreso > 20,000 y deuda < 40 %…".
2. El riesgo de un cliente sube un poquito con cada una de 10 señales (retrasos, deuda, antigüedad…), sumándose.
3. Clientes ubicados en un mapa: compran los que viven en un círculo alrededor de la tienda.

✍️ *Mis apuestas:* 1 → ___ · 2 → ___ · 3 → ___

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Árbol** (reglas con umbrales). 2. **Logística** (señales que se suman: frontera recta). 3. **KNN** (frontera circular: "los cercanos se parecen").

Ahora vamos a comprobarlo con datos.
</details>

---
## 3. ▶️ El torneo: tres conjuntos de datos, tres modelos

Construimos tres conjuntos de datos, uno a la medida de cada apuesta:

In [ ]:
rng = np.random.default_rng(0)

# A) "Reglas": la respuesta sigue umbrales sobre cada variable (como el banco)
X_reglas = rng.uniform(0, 10, (300, 2))
y_reglas = (((X_reglas[:, 0] > 5) & (X_reglas[:, 1] > 3)) | (X_reglas[:, 0] < 2)).astype(int)

# B) "Círculos": una clase rodea a la otra
X_circ, y_circ = make_circles(300, noise=0.1, factor=0.5, random_state=0)

# C) "Suma de señales": 10 variables; la respuesta depende de la suma de 5 de ellas
rng_s = np.random.default_rng(0)
X_suma = rng_s.normal(size=(200, 10))
y_suma = (X_suma[:, :5].sum(axis=1) + rng_s.normal(0, 0.5, 200) > 0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].scatter(*X_reglas.T, c=y_reglas, cmap="coolwarm", s=12); axes[0].set_title("A) Reglas")
axes[1].scatter(*X_circ.T, c=y_circ, cmap="coolwarm", s=12); axes[1].set_title("B) Círculos")
plt.show()
print("C) 'Suma de señales' tiene 10 variables: no se puede dibujar en 2D.")

### 🔮 Ejercicio 3 — Apuesta por un ganador
Vamos a medir el acierto de los tres modelos en los tres conjuntos de datos, con **validación cruzada** (Lección 4.6: se entrena y examina 5 veces con trozos distintos, y se promedia). Antes de ejecutar, completa:

| Datos | Ganador | Perdedor |
|---|---|---|
| A) Reglas | | |
| B) Círculos | | |
| C) Suma de señales | | |

✍️ *Mis apuestas:*

In [ ]:
modelos = {
    "Logística": LogisticRegression(),
    "Árbol (max_depth=4)": DecisionTreeClassifier(max_depth=4, random_state=0),
    "KNN (k=15)": KNeighborsClassifier(n_neighbors=15),
}
conjuntos = {"A) Reglas": (X_reglas, y_reglas), "B) Círculos": (X_circ, y_circ), "C) Suma de señales": (X_suma, y_suma)}

torneo = pd.DataFrame({
    nombre_datos: {nombre_mod: cross_val_score(mod, X, y, cv=5).mean() for nombre_mod, mod in modelos.items()}
    for nombre_datos, (X, y) in conjuntos.items()
}).round(3)
print(torneo)
print("\nGanador por conjunto:", torneo.idxmax().to_dict())

<details><summary><b>👀 Ver respuesta</b></summary>

| Datos | Ganador | Perdedor |
|---|---|---|
| A) Reglas | **Árbol** (≈ 1.00) | Logística (≈ 0.73) |
| B) Círculos | **KNN** (≈ 0.99) | Logística (≈ 0.46, ¡peor que tirar una moneda!) |
| C) Suma de señales | **Logística** (≈ 0.92) | Árbol (≈ 0.73) |

**Tres conjuntos, tres ganadores distintos.** Y la logística, que es la peor en A y B, es la mejor en C. Eso es exactamente lo que dice el teorema: ganar en un problema se paga perdiendo en otro.
</details>

### 📖 ¿Por qué ganó cada uno? Mira cómo "piensan"
La celda dibuja la **frontera de decisión** de cada modelo en los dos conjuntos que podemos ver: la línea donde cambia su respuesta.

In [ ]:
def dibujar_fronteras(X, y, titulo):
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .3, X[:, 0].max() + .3, 200),
                         np.linspace(X[:, 1].min() - .3, X[:, 1].max() + .3, 200))
    for ax, (nombre, mod) in zip(axes, modelos.items()):
        mod.fit(X, y)
        zz = mod.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx, yy, zz, alpha=.3, cmap="coolwarm")
        ax.scatter(*X.T, c=y, cmap="coolwarm", s=8)
        ax.set_title(nombre, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

dibujar_fronteras(X_reglas, y_reglas, "A) Reglas")
dibujar_fronteras(X_circ, y_circ, "B) Círculos")

### ✋ Ejercicio 4 — Lee las fronteras
1. ¿Qué forma tiene **siempre** la frontera de la logística? ¿Por qué no puede con los círculos?
2. ¿Qué forma tiene la frontera del árbol? ¿Por qué encaja tan bien con las reglas?
3. ¿Qué tiene de especial la frontera de KNN?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Una línea recta** (es su apuesta). Ninguna recta puede separar un círculo de un anillo que lo rodea: por eso acierta ~46 %.
2. **Escalones** (líneas verticales y horizontales), porque cada pregunta mira una sola variable. Los datos A se generaron justo así ("x > 5 y y > 3"): su apuesta coincide con la realidad.
3. Se **adapta a cualquier forma**: sigue a los vecinos. Por eso gana con los círculos. Su precio: necesita suficientes datos cercanos y sufre cuando hay muchas variables irrelevantes, como en C (Lección 2.8).
</details>

---
## 4. 📖 ¿Qué pasa si… giramos los datos?

El árbol ganó con las reglas porque los escalones estaban **alineados con los ejes**. ¿Y si tomamos **exactamente los mismos datos** y los **giramos 45°**, como si inclináramos el papel? Los puntos y sus etiquetas no cambian: solo cambia el "punto de vista".

In [ ]:
angulo = np.radians(45)                       # ✏️ prueba 0, 20 y 45 grados
rotacion = np.array([[np.cos(angulo), -np.sin(angulo)], [np.sin(angulo), np.cos(angulo)]])
X_girado = X_reglas @ rotacion

plt.figure(figsize=(4, 4)); plt.scatter(*X_girado.T, c=y_reglas, cmap="coolwarm", s=12)
plt.title("Las mismas reglas, giradas"); plt.show()

### 🔮 Ejercicio 5
Antes de ejecutar: con los datos girados, ¿qué le pasará al acierto del **árbol**? ¿Y al de **KNN**?

✍️ *Mis predicciones:*

In [ ]:
for nombre, mod in modelos.items():
    original = cross_val_score(mod, X_reglas, y_reglas, cv=5).mean()
    girado = cross_val_score(mod, X_girado, y_reglas, cv=5).mean()
    print(f"{nombre:<20} original: {original:.3f}   girado: {girado:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Árbol:** cae de **≈ 0.997** a **≈ 0.79**. Ahora las fronteras son diagonales y el árbol tiene que imitarlas con muchos escalones pequeños; con `max_depth=4` no le alcanza.
- **KNN:** **igual** (≈ 0.94). Girar no cambia qué puntos están cerca de cuáles.
- **Logística:** igual (≈ 0.73): sigue sin poder con una frontera que no es recta.

¡Los **mismos** datos, solo vistos desde otro ángulo, cambian quién gana! El rendimiento de un modelo depende de que sus supuestos encajen con **cómo están representados** los datos. Por eso la ingeniería de características (Lección 2.4) puede valer más que cambiar de modelo.
</details>

---
## 5. 📖 Entonces, ¿cómo elijo un modelo?

Si no hay un modelo perfecto, el método profesional es este:

1. **Empieza por una línea base** muy simple (Lección 4.7). Todo modelo debe ganarle.
2. **Usa lo que sabes del problema** para elegir candidatos razonables: ¿hay reglas?, ¿señales que se suman?, ¿imágenes? (Lección 5.7 y el árbol de decisión de modelos del material de apoyo).
3. **Prueba varios** candidatos con **validación cruzada**, nunca con el examen final.
4. **Elige por la métrica que importa al negocio** (Lección 4.8), no solo por exactitud.
5. Ante un empate, **elige el más simple** y explicable.

> 📝 **Tendencias, no leyes:** en la práctica, los ensambles de árboles (Módulo 6) suelen ganar en datos **tabulares**, y las redes neuronales profundas en **imágenes, audio y texto**. Son buenos puntos de partida, pero el teorema te recuerda que **hay que comprobarlo** en tus datos.

### ✋ Ejercicio 6 — El consultor apresurado
Un consultor te dice: *"No pierdas tiempo probando modelos. Usa siempre una red neuronal: es lo más potente"*. Escribe **dos** argumentos para responderle, usando lo que viste en esta lección.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Por ejemplo:
1. **No hay almuerzo gratis:** ningún modelo es el mejor en todos los problemas. En el torneo, cada conjunto de datos tuvo un ganador distinto, y el "más flexible" no siempre ganó (la logística, el más simple, ganó en C).
2. **Costo y contexto:** con pocos datos o datos en tabla, una red suele necesitar más datos, más ajuste y es más difícil de explicar. Probar 3–4 modelos con validación cruzada cuesta minutos y evita elegir a ciegas.
3. (Extra) Sin una **línea base** y una comparación, no hay forma de saber si la red aporta algo.
</details>

---
## 6. 🚨 Errores comunes y señales de alerta

| 🚨 Error | Por qué es un problema | Qué hacer |
|---|---|---|
| "Siempre uso el modelo X" | Es apostar a ciegas a que tus datos encajan con sus supuestos | Prueba varios candidatos |
| Elegir el modelo por moda o por un blog | El ganador de otro problema no es el tuyo | Compara en **tus** datos |
| Comparar modelos en un solo examen pequeño | El ganador puede serlo por suerte (Lección 1.5) | Validación cruzada (Lección 4.6) |
| Comparar sin línea base | Puedes elegir "el mejor de tres modelos malos" | Incluye siempre una estrategia simple |
| Ajustar solo el modelo e ignorar los datos | Cambiar la representación (características) puede valer más | Ingeniería de características (Lección 2.4) |

### 🔮 Ejercicio 7 — ¿Cuál es el mejor modelo de la lección?
Con todo lo que viste, ¿cuál fue "el mejor modelo" de esta lección?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Ninguno**, y esa es la respuesta correcta. El árbol fue el mejor en A (pero se hundió al girar los datos), KNN en B y la logística en C. "El mejor modelo" solo tiene sentido **para unos datos, con una métrica y bajo unas restricciones** (tiempo, explicabilidad, costo).
</details>

---
## 7. 🧠 Resumen en 5 puntos
1. **No hay almuerzo gratis:** promediando sobre todos los problemas posibles, ningún modelo es mejor que otro.
2. Cada modelo hace una **apuesta** sobre el mundo (sesgo inductivo): rectas, escalones, vecindad…
3. Un modelo gana cuando su apuesta **encaja** con la estructura de los datos; incluso girar los datos puede cambiar el ganador.
4. Por eso se trabaja con **línea base + varios candidatos + validación cruzada + métrica de negocio**.
5. Las "tendencias" (ensambles para tablas, redes para imágenes) son buenos puntos de partida, **no** leyes.

---
## 8. 🏋️ Ejercicios finales

### 8.1 Repaso rápido (opción múltiple)

**1.** ¿Qué dice el teorema de "no hay almuerzo gratis"?
- a) Que el ML siempre cuesta dinero · b) Que, promediando sobre todos los problemas posibles, ningún modelo es mejor que otro · c) Que las redes neuronales siempre ganan · d) Que hay que entrenar gratis

**2.** El "sesgo inductivo" de un modelo es…
- a) Un error de programación · b) Los supuestos que hace para poder generalizar · c) Un prejuicio contra un grupo de personas · d) La tasa de aprendizaje

**3.** ¿Qué modelo tiene fronteras en forma de escalones alineados con los ejes?
- a) Regresión logística · b) Árbol de decisión · c) KNN · d) Ninguno

**4.** Al girar 45° los datos de "reglas", el árbol empeoró y KNN no. ¿Por qué no cambió KNN?
- a) Porque KNN no usa los datos · b) Porque girar no cambia qué puntos son vecinos · c) Por suerte · d) Porque KNN es lineal

**5.** ¿Cuál es la mejor forma de elegir un modelo para un problema nuevo?
- a) Usar el más famoso · b) Usar el que ganó en otro proyecto · c) Comparar varios candidatos con validación cruzada frente a una línea base · d) Usar siempre el más complejo

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** · 2. **b)** (ojo: no confundir con el sesgo social de la Lección 15.4, que es otro concepto) · 3. **b)** · 4. **b)** · 5. **c)**
</details>

### 8.2 Preguntas para pensar (sin ejecutar)

**A.** Si ningún modelo es mejor que otro "en promedio", ¿por qué en la práctica algunos modelos (como los ensambles de árboles) ganan tan a menudo en datos tabulares? ¿Contradice eso el teorema?

**B.** En el Ejercicio 5, el árbol perdió acierto al girar los datos. Imagina que eres el analista y sabes que los datos están "girados" (por ejemplo, la regla real depende de *ingreso − gasto*). ¿Qué harías para devolverle la ventaja al árbol **sin cambiar de modelo**?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**A.** No lo contradice. El teorema promedia sobre **todos** los problemas imaginables, incluidos los absurdos (etiquetas puramente aleatorias). Los problemas reales de negocio tienen una estructura bastante común (relaciones con umbrales, interacciones, variables de distinto tipo) y los ensambles de árboles **apuestan bien** por ese tipo de estructura. Ganan en *ese subconjunto* de problemas, y pagan en otros (por ejemplo, imágenes crudas).

**B.** **Crear una característica nueva** que capture la dirección importante, por ejemplo `ingreso − gasto`. Con esa columna, una sola pregunta del árbol ("¿ingreso − gasto > X?") vuelve a separar bien. Es la **ingeniería de características** (Lección 2.4): cambiar la representación para que los supuestos del modelo encajen.
</details>

### 8.3 🧩 Práctica integradora: el torneo con datos reales

Vas a repetir el torneo con **tres conjuntos de datos reales** incluidos en scikit-learn. 4 pasos cortos; revisa cada respuesta antes de seguir.

| Conjunto | Ejemplos | Características | Qué predice |
|---|---|---|---|
| **Cáncer de mama** | 569 | 30 medidas de células | ¿Tumor benigno o maligno? |
| **Vinos** | 178 | 13 análisis químicos | ¿De cuál de 3 bodegas viene? |
| **Dígitos** | 1,797 | 64 píxeles | ¿Qué dígito (0–9) es? |

> 📝 Aquí la logística y KNN van dentro de un `make_pipeline` con `StandardScaler`, que pone todas las características en la misma escala. Es necesario cuando las columnas tienen unidades muy distintas (lo verás en la Lección 2.3). Por ahora, tómalo como parte del modelo.

**Paso 1 🔮 — Apuesta.** Con lo que sabes de cada modelo, ¿quién crees que ganará en cada conjunto? ¿En cuál crees que el **árbol** sufrirá más?

✍️ *Mis apuestas:*

**Paso 2 ▶️ — El torneo** (tarda unos segundos).

In [ ]:
modelos_reales = {
    "Logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Árbol (max_depth=4)": DecisionTreeClassifier(max_depth=4, random_state=0),
    "KNN (k=15)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
}
datos_reales = {"Cáncer": load_breast_cancer(return_X_y=True), "Vinos": load_wine(return_X_y=True),
                "Dígitos": load_digits(return_X_y=True)}

torneo_real = pd.DataFrame({
    nombre: {m: cross_val_score(mod, X, y, cv=5).mean() for m, mod in modelos_reales.items()}
    for nombre, (X, y) in datos_reales.items()
}).round(3)
print(torneo_real)
print("\nGanador por conjunto:", torneo_real.idxmax().to_dict())

<details><summary><b>👀 Ver respuesta</b></summary>

| | Cáncer | Vinos | Dígitos |
|---|---|---|---|
| Logística | **0.981** | **0.983** | 0.920 |
| Árbol (max_depth=4) | 0.916 | 0.916 | 0.544 |
| KNN (k=15) | 0.961 | 0.955 | **0.933** |

La **logística** gana en cáncer y vinos (muchas medidas que "suman" evidencia), **KNN** gana en dígitos (las imágenes del mismo dígito se parecen píxel a píxel). El **árbol** sufre muchísimo en dígitos: **0.544**.
</details>

**Paso 3 ✏️ — Ayuda al árbol.** Con `max_depth=4` el árbol tiene como mucho 16 hojas para distinguir 10 dígitos y 64 píxeles. Cambia la profundidad y vuelve a ejecutar. ¿Hasta dónde llega? ¿Alcanza a KNN?

In [ ]:
PROFUNDIDAD = 4      # ✏️ prueba 8, 12 y 20
X_dig, y_dig = datos_reales["Dígitos"]
acierto = cross_val_score(DecisionTreeClassifier(max_depth=PROFUNDIDAD, random_state=0), X_dig, y_dig, cv=5).mean()
print(f"Árbol con max_depth={PROFUNDIDAD}: {acierto:.3f}   (KNN: {torneo_real.loc['KNN (k=15)', 'Dígitos']:.3f})")

<details><summary><b>👀 Ver respuesta</b></summary>

Mejora mucho al darle más profundidad (con 8 llega a **0.78**, y a partir de 12 se estanca alrededor de **0.79**), pero **no alcanza** a KNN (0.933). Ajustar los hiperparámetros ayuda, pero no cambia la **apuesta** del modelo: preguntar píxel a píxel por umbrales no es una buena forma de reconocer formas. (Los ensambles de árboles del Módulo 6 mejoran mucho esto, y las redes convolucionales del Módulo 10 lo resuelven casi a la perfección.)
</details>

**Paso 4 ✋ — Conclusiones.**
1. ¿Hubo un modelo que ganara en los tres conjuntos?
2. Si mañana te dan un cuarto conjunto de datos, ¿qué modelo usarías sin probar? (Pregunta trampa.)
3. Escribe en una frase lo que te llevas de esta lección.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No.** La logística ganó dos, KNN uno; el árbol ninguno, aunque en la Sección 3 fue el campeón de las reglas.
2. **Ninguno sin probar.** Puedes elegir **candidatos razonables** según el tipo de datos, pero la decisión se toma comparándolos con validación cruzada frente a una línea base.
3. Por ejemplo: *"No existe el mejor modelo, solo el mejor modelo para estos datos, esta métrica y este contexto, y eso se descubre comparando."*

🎉 **Has terminado las lecciones del Módulo 1.** Ahora toca el proyecto: plantear un problema de negocio real como un proyecto de ML completo.
</details>

---
## 📚 Para profundizar
- 📘 **Artículo original:** D. Wolpert y W. Macready, *No Free Lunch Theorems for Optimization* (1997). Técnico, pero la introducción se lee bien.
- 📘 **scikit-learn:** ejemplo *Classifier comparison* (busca "plot classifier comparison"): las fronteras de 10 modelos en 3 conjuntos de datos, como en la Sección 3.
- 🎮 **Interactivo:** *TensorFlow Playground* (playground.tensorflow.org): prueba los datos "círculo" y "espiral" con una red sin capas ocultas y verás la misma limitación que la logística.

---
⬅️ **Anterior:** [1.5 El ciclo de vida de un proyecto de ML: de la pregunta de negocio al modelo en producción](1.5-ciclo-de-vida-de-un-proyecto.ipynb) · ➡️ **Siguiente:** [1.7 Proyecto del módulo: Gimnasio "FitVida": ¿quién se va a dar de baja? 🏋️](1.7-proyecto-del-modulo.md)